# Live Liver Donor Expert System (Extended Multi-Model Edition)
Rule-based clinical screening (Hard Exclusions & Soft Warnings) followed by a **4-model ML ensemble** (KNN, Logistic Regression, Naive Bayes, Decision Tree) with explainability, cohort baseline summaries, model-by-model reasoning, and what-if sensitivity analysis.

## Model 1: KNN + Data (Lab 1)

In [9]:
import numpy as np

# Historical Liver Donor Dataset
# Features: [Age, BMI, Smoking Habit (0=No, 1=Yes), General Health Score (1-5)]
X_train = np.array([
    [28, 22.5, 0, 5],
    [55, 34.0, 1, 2],
    [35, 24.0, 0, 4],
    [45, 31.5, 1, 2],
    [24, 21.0, 0, 5],
    [58, 29.0, 0, 3],
    [40, 36.0, 1, 1],
    [30, 23.5, 0, 4],
])
y_train = np.array([1, 0, 1, 0, 1, 1, 0, 1])

class Scaler:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1e-8

    def transform(self, X):
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

class KNN:

  def __init__(self, k=3):
    self.k = k
    self.scaler = Scaler()

  def fit(self, X, y):
    self.raw_X = X
    self.X_train = self.scaler.fit_transform(X)
    self.y_train = y


  def _euclidean_distance(self, x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

  def predict(self, x_new):
    distances = []
    x_new = self.scaler.transform(x_new)
    for x_train in self.X_train:
      dist = self._euclidean_distance(x_new, x_train)
      distances.append(dist)

    k_indices = np.argsort(distances)[: self.k]
    k_nearest_labels = self.y_train[k_indices]

    unique_labels, counts = np.unique(k_nearest_labels, return_counts=True)
    return unique_labels[np.argmax(counts)]

  def explain(self, x_new):
    """Returns inverse-distance probability and the k nearest historical donor cases."""
    x_scaled = self.scaler.transform(x_new)
    distances = np.array([self._euclidean_distance(x_scaled, x_tr) for x_tr in self.X_train])
    k_indices = np.argsort(distances)[:self.k]
    k_dists = distances[k_indices]
    k_labels = self.y_train[k_indices]

    weights = 1.0 / (k_dists + 1e-5)
    p_elig = float(np.sum(weights[k_labels == 1]) / np.sum(weights))

    neighbors = [
        {
            'case_id': int(idx) + 1,
            'features': self.raw_X[idx],
            'label': int(self.y_train[idx]),
            'distance': round(float(dist), 3),
        }
        for idx, dist in zip(k_indices, k_dists)
    ]
    return p_elig, neighbors


knn = KNN(k=3)
knn.fit(X_train, y_train)
print('System Initialized: Historical data loaded & KNN model ready.')

System Initialized: Historical data loaded & KNN model ready.


## Model 2: Logistic Regression (Lab 4)

In [10]:
class LogisticRegression:
    """Logistic regression trained with gradient descent + L2 regularization (from Lab 4)."""

    def __init__(self, lr=0.1, alpha=1.0, n_iter=5000):
        self.lr = lr
        self.alpha = alpha
        self.n_iter = n_iter
        self.scaler = Scaler()

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = self.scaler.fit_transform(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=float)
        N = len(X)
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        for _ in range(self.n_iter):
            p = self._sigmoid(X @ self.w + self.b)
            e = p - y
            db = e.sum() / N
            dw = (X.T @ e) / N + (self.alpha / N) * self.w
            self.b -= self.lr * db
            self.w -= self.lr * dw

    def predict_proba(self, x):
        x = self.scaler.transform(np.asarray(x, dtype=float))
        return float(self._sigmoid(x @ self.w + self.b))

    def predict(self, x, thr=0.5):
        return int(self.predict_proba(x) >= thr)

    def explain_contributions(self, x):
        """Returns standardized log-odds contribution (w_j * z_j) for each feature."""
        z = self.scaler.transform(np.asarray(x, dtype=float))
        return z * self.w


logreg = LogisticRegression()
logreg.fit(X_train, y_train)
print('Logistic Regression trained. Feature weights (standardized):',
      np.round(logreg.w, 2))

Logistic Regression trained. Feature weights (standardized): [-0.11 -0.61 -0.94  0.61]


## Models 3 & 4: Naive Bayes (Lab 5) and Decision Tree (Lab 2)
These two work on categories, so the numeric features are first grouped into bins.

In [11]:
FEATURE_NAMES = ['Age', 'BMI', 'Smoking', 'Health']

# Human-readable names for each category (used when explaining the tree)
CATEGORY_LABELS = {
    'Age':     {1: '<=30', 2: '31-45', 3: '>45'},
    'BMI':     {1: '<25 (normal)', 2: '25-30 (overweight)', 3: '>30 (obese)'},
    'Smoking': {0: 'non-smoker', 1: 'smoker'},
    'Health':  {1: 'poor (1-2)', 2: 'average (3)', 3: 'good (4-5)'},
}

def categorize_features(sample):
    """Turns [age, bmi, smoking, health] into categories (as in Lab 1)."""
    age, bmi, smoking, health = sample

    cat_age = 1 if age <= 30 else 2 if age <= 45 else 3
    cat_bmi = 1 if bmi < 25 else 2 if bmi <= 30 else 3
    cat_health = 1 if health <= 2 else 2 if health == 3 else 3

    return [cat_age, cat_bmi, int(smoking), cat_health]

X_train_cat = [categorize_features(x) for x in X_train]

In [12]:
from collections import Counter
import math

class CategoricalNaiveBayes:
    """Naive Bayes for categorical features with Laplace smoothing (from Lab 5)."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        y = [int(v) for v in y]
        n_features = len(X[0])
        self.classes = sorted(set(y))
        self.class_counts = Counter(y)
        self.priors = {c: self.class_counts[c] / len(y) for c in self.classes}
        self.values = [set(row[j] for row in X) for j in range(n_features)]
        self.counts = {c: [Counter() for _ in range(n_features)] for c in self.classes}
        for row, label in zip(X, y):
            for j, v in enumerate(row):
                self.counts[label][j][v] += 1

    def _likelihood(self, c, j, v):
        numerator = self.counts[c][j][v] + self.alpha
        denominator = self.class_counts[c] + self.alpha * len(self.values[j])
        return numerator / denominator

    def predict_proba(self, row):
        """Returns {class: posterior probability}."""
        log_scores = {}
        for c in self.classes:
            score = math.log(self.priors[c])
            for j, v in enumerate(row):
                score += math.log(self._likelihood(c, j, v))
            log_scores[c] = score
        m = max(log_scores.values())
        exp = {c: math.exp(s - m) for c, s in log_scores.items()}
        total = sum(exp.values())
        return {c: v / total for c, v in exp.items()}

    def predict(self, row):
        proba = self.predict_proba(row)
        return max(proba, key=proba.get)


nb = CategoricalNaiveBayes()
nb.fit(X_train_cat, y_train)
print('Naive Bayes trained on categorized features.')

Naive Bayes trained on categorized features.


In [13]:
class DecisionTree:
    """ID3 decision tree using entropy / information gain (from Lab 2)."""

    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    @staticmethod
    def _entropy(y):
        n = len(y)
        return -sum((c / n) * math.log2(c / n) for c in Counter(y).values())

    def _gain(self, X, y, j):
        n = len(y)
        remainder = 0.0
        for v in set(row[j] for row in X):
            y_v = [label for row, label in zip(X, y) if row[j] == v]
            remainder += (len(y_v) / n) * self._entropy(y_v)
        return self._entropy(y) - remainder

    def _build(self, X, y, features, depth):
        majority = Counter(y).most_common(1)[0][0]
        if len(set(y)) == 1 or not features or (self.max_depth is not None and depth >= self.max_depth):
            return {'leaf': majority}

        best = max(features, key=lambda j: self._gain(X, y, j))
        children = {}
        for v in set(row[best] for row in X):
            idx = [i for i, row in enumerate(X) if row[best] == v]
            children[v] = self._build(
                [X[i] for i in idx], [y[i] for i in idx],
                [f for f in features if f != best], depth + 1,
            )
        return {'feature': best, 'majority': majority, 'children': children}

    def fit(self, X, y):
        y = [int(v) for v in y]
        self.root = self._build(X, y, list(range(len(X[0]))), 0)

    def predict(self, row):
        node = self.root
        while 'leaf' not in node:
            v = row[node['feature']]
            if v not in node['children']:   # category never seen in this branch
                return node['majority']
            node = node['children'][v]
        return node['leaf']

    def trace_path(self, row):
        """Returns the human-readable decision path taken for a categorized sample."""
        node = self.root
        steps = []
        while 'leaf' not in node:
            j = node['feature']
            name = FEATURE_NAMES[j]
            v = row[j]
            label_str = CATEGORY_LABELS[name].get(v, str(v))
            steps.append(f"{name} = {label_str}")
            if v not in node['children']:
                steps.append("(unseen branch -> majority fallback)")
                break
            node = node['children'][v]
        return " -> ".join(steps)

    def print_tree(self, node=None, indent=''):
        node = node or self.root
        if 'leaf' in node:
            print(f"{indent}-> {'ELIGIBLE' if node['leaf'] == 1 else 'HIGH RISK'}")
            return
        name = FEATURE_NAMES[node['feature']]
        for v, child in sorted(node['children'].items()):
            print(f"{indent}{name} = {CATEGORY_LABELS[name][v]}")
            self.print_tree(child, indent + '    ')


tree = DecisionTree()
tree.fit(X_train_cat, y_train)
print('Decision Tree trained. Learned rules:')
tree.print_tree()

Decision Tree trained. Learned rules:
BMI = <25 (normal)
    -> ELIGIBLE
BMI = 25-30 (overweight)
    -> ELIGIBLE
BMI = >30 (obese)
    -> HIGH RISK


## Rule-based screening (Hard Exclusions + Soft Warnings)

In [14]:
def evaluate_liver_donor_rules(
    age, bmi, smoking, has_infection, has_chronic_disease
):
  """Original mandatory medical exclusion criteria for live liver donation."""

  if age < 18 or age > 60:
    return (
        False,
        f'Rejected by Rules: Age {age} is outside the safe range (18-60 years).'
        ' Live donors generally need to be adults under 60.',
    )

  if bmi < 18.5 or bmi > 32.0:
    return (
        False,
        f'Rejected by Rules: BMI {bmi:.1f} is outside the acceptable range'
        ' (18.5 - 32.0).',
    )

  if has_infection:
    return (
        False,
        'Rejected by Rules: Active infection detected. Cannot proceed while'
        ' fighting an infection.',
    )

  if has_chronic_disease:
    return (
        False,
        'Rejected by Rules: History of chronic disease disqualifies live'
        ' organ donation.',
    )

  return True, 'Passed: All hard medical screening filters cleared.'


def classify_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25.0:
        return 'Normal Weight'
    elif bmi < 30.0:
        return 'Overweight'
    return 'Obese'


def audit_donor_rules_detailed(age, bmi, smoking, has_infection, has_chronic_disease, health_score):
    """Full multi-rule audit without early exit: separates Hard Exclusions from Soft Warnings."""
    orig_passed, orig_msg = evaluate_liver_donor_rules(
        age, bmi, smoking, has_infection, has_chronic_disease
    )

    exclusions = []
    warnings = []
    passed = []
    recommendations = []

    # 1. Age
    if age < 18 or age > 60:
        exclusions.append(f"Age {age} is outside safe donor bounds (18–60 years).")
    elif age > 50:
        warnings.append(f"Age {age} is in the upper bracket (51–60); extra liver volumetry advised.")
    else:
        passed.append(f"Age ({age} yrs) is in optimal donor range.")

    # 2. BMI
    bmi_cat = classify_bmi(bmi)
    if bmi < 18.5 or bmi > 32.0:
        exclusions.append(f"BMI {bmi:.1f} ({bmi_cat}) exceeds safety bounds (18.5–32.0).")
        if bmi > 32.0:
            recommendations.append("Reduce BMI below 30.0 to lower hepatic steatosis (fatty liver) risk.")
    elif bmi > 28.0:
        warnings.append(f"BMI {bmi:.1f} ({bmi_cat}) is borderline; MRI/biopsy fat check recommended.")
        recommendations.append("Target a BMI < 25.0 for optimal post-donation liver regeneration.")
    else:
        passed.append(f"BMI ({bmi:.1f} - {bmi_cat}) is within safe surgical limits.")

    # 3. Active Infection
    if has_infection:
        exclusions.append("Active infection detected; donation is blocked until full recovery.")
        recommendations.append("Re-screen 4 weeks after infection resolution and normal blood work.")
    else:
        passed.append("No active infections reported.")

    # 4. Chronic Disease
    if has_chronic_disease:
        exclusions.append("History of chronic systemic/hepatic disease disqualifies live donation.")
    else:
        passed.append("No disqualifying chronic conditions reported.")

    # 5. Soft Clinical Flags (Smoking & Self-Reported Health)
    if smoking == 1:
        warnings.append("Active smoker: elevated anesthetic and post-op vascular complication risk.")
        recommendations.append("Smoking cessation is required at least 6–8 weeks prior to surgery.")
    else:
        passed.append("Non-smoker status verified.")

    if health_score <= 2:
        warnings.append(f"Low self-reported health score ({health_score}/5) requires full cardiopulmonary workup.")

    return {
        'orig_passed': orig_passed,
        'orig_msg': orig_msg,
        'passed_hard_rules': len(exclusions) == 0,
        'bmi_category': bmi_cat,
        'exclusions': exclusions,
        'warnings': warnings,
        'passed': passed,
        'recommendations': recommendations,
    }

## Multi-model prediction, cohort summary & what-if analysis

In [15]:
MODEL_NAMES = ['KNN (k=3)', 'Logistic Regression', 'Naive Bayes', 'Decision Tree']
ELIGIBLE_COHORT_MEAN = np.mean(X_train[y_train == 1], axis=0)
HIGH_RISK_COHORT_MEAN = np.mean(X_train[y_train == 0], axis=0)

def multi_model_predict(age, bmi, smoking, health_score):
    """Asks every ML model for its opinion, probabilities, and model-specific explanations."""
    x = np.array([age, bmi, smoking, health_score], dtype=float)
    x_cat = categorize_features([age, bmi, smoking, health_score])

    knn_p_elig, knn_neighbors = knn.explain(x)
    lr_p_elig = logreg.predict_proba(x)
    lr_contrib = logreg.explain_contributions(x)
    nb_p_elig = nb.predict_proba(x_cat)[1]
    dt_path = tree.trace_path(x_cat)

    votes = {
        'KNN (k=3)': int(knn.predict(x)),
        'Logistic Regression': logreg.predict(x),
        'Naive Bayes': int(nb.predict(x_cat)),
        'Decision Tree': int(tree.predict(x_cat)),
    }
    p_eligible = {
        'KNN (k=3)': knn_p_elig,
        'Logistic Regression': lr_p_elig,
        'Naive Bayes': nb_p_elig,
    }

    n_yes = sum(votes.values())
    n_total = len(votes)
    if n_yes > n_total / 2:
        verdict = 'ELIGIBLE'
    elif n_yes < n_total / 2:
        verdict = 'HIGH_RISK'
    else:
        verdict = 'INCONCLUSIVE'   # tie: safest option is a human decision

    # Ensemble average probability across probabilistic models
    mean_p_elig = float(np.mean(list(p_eligible.values())))

    return {
        'votes': votes,
        'p_eligible': p_eligible,
        'mean_p_elig': mean_p_elig,
        'n_yes': n_yes,
        'n_total': n_total,
        'verdict': verdict,
        'x_cat': x_cat,
        'knn_neighbors': knn_neighbors,
        'lr_contrib': lr_contrib,
        'dt_path': dt_path,
    }


def run_what_if_simulations(age, bmi, smoking, health_score):
    """Simulates how modifiable lifestyle changes affect the 4-model ensemble."""
    scenarios = []
    base = multi_model_predict(age, bmi, smoking, health_score)

    if smoking == 1:
        sim = multi_model_predict(age, bmi, 0, health_score)
        scenarios.append((
            'Quit Smoking (Smoking -> 0)',
            sim['n_yes'], sim['n_total'], sim['mean_p_elig'] - base['mean_p_elig'], sim['verdict']
        ))

    if bmi > 24.5:
        sim = multi_model_predict(age, 24.0, smoking, health_score)
        scenarios.append((
            'Lower BMI to 24.0 (Normal Weight)',
            sim['n_yes'], sim['n_total'], sim['mean_p_elig'] - base['mean_p_elig'], sim['verdict']
        ))

    if health_score < 4:
        sim = multi_model_predict(age, bmi, smoking, 4)
        scenarios.append((
            'Improve Health Score to 4/5',
            sim['n_yes'], sim['n_total'], sim['mean_p_elig'] - base['mean_p_elig'], sim['verdict']
        ))

    return scenarios


def print_ensemble_report(applicant, rule_report, result):
    print('\n' + '=' * 70)
    print('              LIVE LIVER DONOR EXPERT SYSTEM SUMMARY')
    print('=' * 70)

    # 1. Applicant Profile & Categorization
    cat_vals = categorize_features([
        applicant['age'], applicant['bmi'], applicant['smoking'], applicant['health_score']
    ])
    print('\n[1] APPLICANT PROFILE & FEATURE BINNING')
    print(f"  • Age          : {applicant['age']} yrs  (Bin: {CATEGORY_LABELS['Age'][cat_vals[0]]})")
    print(f"  • Weight/Height: {applicant['weight']} kg / {applicant['height']} m")
    print(f"  • BMI          : {applicant['bmi']:.1f} [{rule_report['bmi_category']}]  (Bin: {CATEGORY_LABELS['BMI'][cat_vals[1]]})")
    print(f"  • Smoking      : {'Yes' if applicant['smoking'] else 'No'}  (Bin: {CATEGORY_LABELS['Smoking'][cat_vals[2]]})")
    print(f"  • Health Score : {applicant['health_score']}/5  (Bin: {CATEGORY_LABELS['Health'][cat_vals[3]]})")

    # 2. Rule Engine Audit
    print('\n[2] RULE-BASED CLINICAL SCREENING AUDIT')
    for msg in rule_report['passed']:
        print(f'  [PASS] {msg}')
    for msg in rule_report['warnings']:
        print(f'  [WARN] {msg}')
    for msg in rule_report['exclusions']:
        print(f'  [FAIL] {msg}')

    # 3. Cohort Value Summary
    print('\n[3] COHORT VALUE SUMMARY (YOUR VALUES VS. HISTORICAL AVERAGES)')
    print(f"  {'Feature':<10} | {'You':>7} | {'Eligible Avg':>12} | {'High-Risk Avg':>13} | {'Diff vs Elig':>12}")
    print('  ' + '-' * 64)
    vals = [applicant['age'], applicant['bmi'], applicant['smoking'], applicant['health_score']]
    for i, name in enumerate(FEATURE_NAMES):
        diff = vals[i] - ELIGIBLE_COHORT_MEAN[i]
        sign = '+' if diff > 0 else ''
        print(
            f"  {name:<10} | {vals[i]:>7.1f} | {ELIGIBLE_COHORT_MEAN[i]:>12.1f} | "
            f"{HIGH_RISK_COHORT_MEAN[i]:>13.1f} | {sign}{diff:>11.1f}"
        )

    if result is None:
        print('\n[4] ML ENSEMBLE & FINAL DECISION')
        print(f"  Status: DISQUALIFIED ({len(rule_report['exclusions'])} hard rule(s) failed).")
        print('  Note  : ML models bypassed due to mandatory medical exclusions.')
        if rule_report['recommendations']:
            print('\n  Recommended Next Steps:')
            for rec in rule_report['recommendations']:
                print(f'    -> {rec}')
        print('=' * 70)
        return

    # 4. Multi-Model Vote
    print('\n[4] ML MODEL BLOCK (4-MODEL ENSEMBLE VOTE)')
    for name in MODEL_NAMES:
        vote = 'ELIGIBLE' if result['votes'][name] == 1 else 'HIGH RISK'
        extra = ''
        if name in result['p_eligible']:
            label_tag = 'weighted' if name == 'KNN (k=3)' else 'posterior'
            extra = f"  (P(eligible) = {result['p_eligible'][name]:.0%} [{label_tag}])"
        print(f'  {name:<20} -> {vote:<10}{extra}')

    print(f"\n  Ensemble Tally     : {result['n_yes']} / {result['n_total']} models voted ELIGIBLE")
    print(f"  Mean P(eligible)   : {result['mean_p_elig']:.1%}")

    # 5. Model-by-Model Reasoning
    print('\n[5] MODEL EXPLAINABILITY')
    print(f"  • Decision Tree Path : {result['dt_path']}")
    contrib_str = ', '.join(
        f"{name}={val:+.2f}" for name, val in zip(FEATURE_NAMES, result['lr_contrib'])
    )
    print(f"  • LogReg Log-Odds    : {contrib_str}  (+ favors eligible, - favors risk)")
    print('  • KNN Nearest Cases  :')
    for n in result['knn_neighbors']:
        f_arr = n['features']
        lbl = 'Eligible' if n['label'] == 1 else 'High Risk'
        print(
            f"      - Case #{n['case_id']} [{lbl}] (Dist: {n['distance']}): "
            f"Age={int(f_arr[0])}, BMI={f_arr[1]}, Smoke={int(f_arr[2])}, Health={int(f_arr[3])}"
        )

    # 6. What-If Sensitivity Analysis
    scenarios = run_what_if_simulations(
        applicant['age'], applicant['bmi'], applicant['smoking'], applicant['health_score']
    )
    if scenarios:
        print('\n[6] WHAT-IF SENSITIVITY ANALYSIS (MODIFIABLE FACTORS)')
        for desc, yes_cnt, tot_cnt, delta_p, new_verdict in scenarios:
            sign = '+' if delta_p >= 0 else ''
            print(f"  • If {desc:<34} -> {yes_cnt}/{tot_cnt} votes ({sign}{delta_p:.0%} prob) => {new_verdict}")

    # 7. Final Verdict & Recommendations
    print('\n[7] FINAL DECISION & ACTIONABLE SUMMARY')
    unanimous = result['n_yes'] in (0, result['n_total'])
    if result['verdict'] == 'ELIGIBLE':
        print(
            '  Result: 🟢 ELIGIBLE DONOR PROFILE. Based on historical data similarity,'
            ' your profile matches successful live liver donors. (Note: Clinical'
            ' lab tests are still required).'
        )
    elif result['verdict'] == 'HIGH_RISK':
        print(
            '  Result: 🔴 HIGH RISK PROFILE. While you passed initial safety rules,'
            ' historical patterns suggest potential elevated risks for this type'
            ' of donation profile.'
        )
    else:
        print(
            '  Result: 🟡 INCONCLUSIVE. The models are split 50/50 on this profile,'
            ' so it should be reviewed by a medical professional.'
        )

    if rule_report['warnings']:
        print(f"  Clinical Flags: {len(rule_report['warnings'])} soft warning(s) require physician review.")
    if not unanimous and result['verdict'] != 'INCONCLUSIVE':
        print('  Model Note    : The models did not all agree, so treat this result with extra caution.')

    if rule_report['recommendations']:
        print('\n  Recommended Next Steps:')
        for rec in rule_report['recommendations']:
            print(f'    -> {rec}')
    print('=' * 70)


def assess_donor(age, weight_kg, height_m, smoking, has_infection,
                 has_chronic_disease, health_score):
    """Full pipeline: comprehensive rule audit first, then the 4-model ML ensemble."""
    bmi = weight_kg / (height_m ** 2)
    applicant = {
        'age': age,
        'weight': weight_kg,
        'height': height_m,
        'bmi': bmi,
        'smoking': smoking,
        'health_score': health_score,
    }

    rule_report = audit_donor_rules_detailed(
        age, bmi, smoking, has_infection, has_chronic_disease, health_score
    )

    if not rule_report['passed_hard_rules']:
        print_ensemble_report(applicant, rule_report, None)
        return None

    result = multi_model_predict(age, bmi, smoking, health_score)
    print_ensemble_report(applicant, rule_report, result)
    return result

## Questionnaire

In [16]:
def ask_yes_no(prompt):
    """Keeps asking until the answer is exactly yes or no."""
    while True:
        answer = input(prompt).strip().lower()
        if answer in ('yes', 'y'):
            return True
        if answer in ('no', 'n'):
            return False
        print("  Please answer 'yes' or 'no'.")


print('=== LIVE LIVER DONOR ELIGIBILITY QUESTIONNAIRE ===\n')

try:
  age = int(input('Enter your age (years): '))
  weight_kg = float(input('Enter your weight (kg): '))
  height_m = float(input('Enter your height (meters, e.g., 1.75): '))
  if weight_kg <= 0 or height_m <= 0:
    raise ValueError

  smoking = 1 if ask_yes_no('Do you smoke regularly? (yes/no): ') else 0
  has_infection = ask_yes_no('Do you have any active infections currently? (yes/no): ')
  has_chronic_disease = ask_yes_no('Do you have chronic diseases (diabetes, hepatitis)? (yes/no): ')

  health_score = int(
      input('Rate your overall self-reported health from 1 (poor) to 5 (excellent): ')
  )
  if not 1 <= health_score <= 5:
    raise ValueError

  assess_donor(age, weight_kg, height_m, smoking, has_infection,
               has_chronic_disease, health_score)

except ValueError:
  print(
      '\nInvalid input format. Please restart the cell and enter numbers where'
      ' requested (health score must be between 1 and 5).'
  )

=== LIVE LIVER DONOR ELIGIBILITY QUESTIONNAIRE ===

Enter your age (years): 21
Enter your weight (kg): 99
Enter your height (meters, e.g., 1.75): 1.84
Do you smoke regularly? (yes/no): n
Do you have any active infections currently? (yes/no): n
Do you have chronic diseases (diabetes, hepatitis)? (yes/no): n
Rate your overall self-reported health from 1 (poor) to 5 (excellent): 4

              LIVE LIVER DONOR EXPERT SYSTEM SUMMARY

[1] APPLICANT PROFILE & FEATURE BINNING
  • Age          : 21 yrs  (Bin: <=30)
  • Weight/Height: 99.0 kg / 1.84 m
  • BMI          : 29.2 [Overweight]  (Bin: 25-30 (overweight))
  • Smoking      : No  (Bin: non-smoker)
  • Health Score : 4/5  (Bin: good (4-5))

[2] RULE-BASED CLINICAL SCREENING AUDIT
  [PASS] Age (21 yrs) is in optimal donor range.
  [PASS] No active infections reported.
  [PASS] No disqualifying chronic conditions reported.
  [PASS] Non-smoker status verified.
  [WARN] BMI 29.2 (Overweight) is borderline; MRI/biopsy fat check recommended.
